# PD — Feature Engineering

Create transparent, reproducible predictors and document transformations.


# PD — Feature Engineering

Preparación de variables para el modelo de Probability of Default (PD).

En esta etapa se realizarán:

- tratamiento de valores faltantes;
- tratamiento de valores extremos;
- transformación de variables;
- creación de indicadores de missing;
- creación de variables derivadas;
- selección preliminar de variables;
- preparación del dataset final para modelamiento.

El objetivo es obtener un dataset consistente y adecuado para entrenar posteriormente los modelos de Regresión Logística y XGBoost.

In [1]:
# ============================================
# PD — Feature Engineering
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ruta del dataset original
DATA_PATH = "../data/raw/cs-training.csv"

# Cargar dataset
df = pd.read_csv(DATA_PATH)

print("Dataset cargado correctamente.")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

Dataset cargado correctamente.
Filas: 150,000
Columnas: 12


In [2]:
# ============================================
# Indicadores de valores faltantes
# ============================================

# Indicador de MonthlyIncome faltante
df["MonthlyIncome_missing"] = df["MonthlyIncome"].isna().astype(int)

# Indicador de NumberOfDependents faltante
df["NumberOfDependents_missing"] = df["NumberOfDependents"].isna().astype(int)

# Resumen
missing_indicators = pd.DataFrame({
    "variable": [
        "MonthlyIncome_missing",
        "NumberOfDependents_missing"
    ],
    "count": [
        df["MonthlyIncome_missing"].sum(),
        df["NumberOfDependents_missing"].sum()
    ]
})

missing_indicators["percentage"] = (
    missing_indicators["count"] / len(df) * 100
)

display(missing_indicators)

,variable,count,percentage
0,MonthlyIncome_missing,29731,19.820667
1,NumberOfDependents_missing,3924,2.616000


In [3]:
# ============================================
# Imputación de MonthlyIncome
# ============================================

# Mediana calculada únicamente sobre los valores disponibles
monthly_income_median = df["MonthlyIncome"].median()

print(f"Mediana de MonthlyIncome: {monthly_income_median:.2f}")

# Imputación
df["MonthlyIncome_imputed"] = (
    df["MonthlyIncome"]
    .fillna(monthly_income_median)
)

# Verificación
print(
    f"Valores faltantes antes: {df['MonthlyIncome'].isna().sum():,}"
)

print(
    f"Valores faltantes después: "
    f"{df['MonthlyIncome_imputed'].isna().sum():,}"
)

Mediana de MonthlyIncome: 5400.00
Valores faltantes antes: 29,731
Valores faltantes después: 0


In [4]:
# ============================================
# Imputación de NumberOfDependents
# ============================================

# Mediana calculada sobre los valores disponibles
dependents_median = df["NumberOfDependents"].median()

print(f"Mediana de NumberOfDependents: {dependents_median:.2f}")

# Imputación
df["NumberOfDependents_imputed"] = (
    df["NumberOfDependents"]
    .fillna(dependents_median)
)

# Verificación
print(
    f"Valores faltantes antes: "
    f"{df['NumberOfDependents'].isna().sum():,}"
)

print(
    f"Valores faltantes después: "
    f"{df['NumberOfDependents_imputed'].isna().sum():,}"
)

Mediana de NumberOfDependents: 0.00
Valores faltantes antes: 3,924
Valores faltantes después: 0


In [5]:
# ============================================
# Tratamiento de valores especiales 96 y 98
# ============================================

# Copia de las variables de mora
late_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]

# Indicadores de valores especiales
for col in late_cols:
    df[f"{col}_special_96"] = (df[col] == 96).astype(int)
    df[f"{col}_special_98"] = (df[col] == 98).astype(int)

# Convertir 96 y 98 en valores faltantes
for col in late_cols:
    df[col] = df[col].replace([96, 98], np.nan)

# Verificación
special_check = pd.DataFrame({
    "variable": late_cols,
    "value_96_after": [
        (df[col] == 96).sum()
        for col in late_cols
    ],
    "value_98_after": [
        (df[col] == 98).sum()
        for col in late_cols
    ],
    "missing_after": [
        df[col].isna().sum()
        for col in late_cols
    ]
})

display(special_check)

,variable,value_96_after,value_98_after,missing_after
0,NumberOfTime30-59DaysPastDueNotWorse,0,0,269
1,NumberOfTimes90DaysLate,0,0,269
2,NumberOfTime60-89DaysPastDueNotWorse,0,0,269


In [6]:
# ============================================
# Imputación de variables de mora
# ============================================

for col in late_cols:
    df[f"{col}_imputed"] = df[col].fillna(0)

# Verificación
late_imputation_check = pd.DataFrame({
    "variable": late_cols,
    "missing_before": [
        df[col].isna().sum()
        for col in late_cols
    ],
    "missing_after": [
        df[f"{col}_imputed"].isna().sum()
        for col in late_cols
    ],
    "min_after": [
        df[f"{col}_imputed"].min()
        for col in late_cols
    ]
})

display(late_imputation_check)

,variable,missing_before,missing_after,min_after
0,NumberOfTime30-59DaysPastDueNotWorse,269,0,0.0
1,NumberOfTimes90DaysLate,269,0,0.0
2,NumberOfTime60-89DaysPastDueNotWorse,269,0,0.0


In [7]:
# ============================================
# Tratamiento de valores inválidos de edad
# ============================================

# Crear copia de edad
df["age_clean"] = df["age"]

# Marcar edades inválidas
df["age_invalid"] = (
    (df["age"] < 18) |
    (df["age"] > 100)
).astype(int)

# Convertir edades inválidas en NaN
df.loc[df["age_invalid"] == 1, "age_clean"] = np.nan

# Imputar con la mediana de las edades válidas
age_median = df.loc[df["age_invalid"] == 0, "age_clean"].median()

df["age_clean"] = df["age_clean"].fillna(age_median)

# Verificación
print(f"Mediana de edad válida: {age_median:.2f}")
print(f"Edades inválidas detectadas: {df['age_invalid'].sum():,}")
print(f"Faltantes después: {df['age_clean'].isna().sum():,}")

display(
    df.loc[
        df["age_invalid"] == 1,
        ["age", "age_clean", "age_invalid", "SeriousDlqin2yrs"]
    ]
)

Mediana de edad válida: 52.00
Edades inválidas detectadas: 14
Faltantes después: 0


,age,age_clean,age_invalid,SeriousDlqin2yrs
7763,101,52.0,1,0
19884,103,52.0,1,0
25561,102,52.0,1,0
40007,107,52.0,1,0
56761,105,52.0,1,0
57967,103,52.0,1,0
65695,0,52.0,1,0
90937,102,52.0,1,0
93813,101,52.0,1,0
96450,102,52.0,1,0


In [8]:
# ============================================
# Tratamiento de DebtRatio
# ============================================

# Crear copia
df["DebtRatio_clean"] = df["DebtRatio"]

# Indicador de valores extremos
df["DebtRatio_extreme"] = (
    df["DebtRatio"] > 100
).astype(int)

# Limitar los valores extremos al percentil 99
debt_ratio_p99 = df["DebtRatio"].quantile(0.99)

df["DebtRatio_clean"] = df["DebtRatio_clean"].clip(
    upper=debt_ratio_p99
)

# Verificación
print(f"P99 de DebtRatio: {debt_ratio_p99:.2f}")
print(
    f"Valores originalmente > 100: "
    f"{df['DebtRatio_extreme'].sum():,}"
)
print(
    f"Máximo antes: {df['DebtRatio'].max():,.2f}"
)
print(
    f"Máximo después: {df['DebtRatio_clean'].max():,.2f}"
)

display(
    df[
        [
            "DebtRatio",
            "DebtRatio_clean",
            "DebtRatio_extreme"
        ]
    ].describe()
)

P99 de DebtRatio: 4979.04
Valores originalmente > 100: 24,380
Máximo antes: 329,664.00
Máximo después: 4,979.04


,DebtRatio,DebtRatio_clean,DebtRatio_extreme
count,150000.000000,150000.000000,150000.000000
mean,353.005076,316.548869,0.162533
std,2037.818523,906.962222,0.368941
min,0.000000,0.000000,0.000000
25%,0.175074,0.175074,0.000000
50%,0.366508,0.366508,0.000000
75%,0.868254,0.868254,0.000000
max,329664.000000,4979.040000,1.000000


In [9]:
# ============================================
# Tratamiento de RevolvingUtilization
# ============================================

# Crear copia
df["Utilization_clean"] = df["RevolvingUtilizationOfUnsecuredLines"]

# Indicador de valores extremos
df["Utilization_extreme"] = (
    df["RevolvingUtilizationOfUnsecuredLines"] > 10
).astype(int)

# Calcular P99
utilization_p99 = (
    df["RevolvingUtilizationOfUnsecuredLines"]
    .quantile(0.99)
)

# Limitar valores extremos al P99
df["Utilization_clean"] = (
    df["Utilization_clean"]
    .clip(upper=utilization_p99)
)

# Verificación
print(f"P99 de Utilization: {utilization_p99:.4f}")

print(
    f"Valores originalmente > 10: "
    f"{df['Utilization_extreme'].sum():,}"
)

print(
    f"Máximo antes: "
    f"{df['RevolvingUtilizationOfUnsecuredLines'].max():,.2f}"
)

print(
    f"Máximo después: "
    f"{df['Utilization_clean'].max():,.4f}"
)

display(
    df[
        [
            "RevolvingUtilizationOfUnsecuredLines",
            "Utilization_clean",
            "Utilization_extreme"
        ]
    ].describe()
)

P99 de Utilization: 1.0930
Valores originalmente > 10: 241
Máximo antes: 50,708.00
Máximo después: 1.0930


,RevolvingUtilizationOfUnsecuredLines,Utilization_clean,Utilization_extreme
count,150000.000000,150000.000000,150000.000000
mean,6.048438,0.320496,0.001607
std,249.755371,0.352152,0.040051
min,0.000000,0.000000,0.000000
25%,0.029867,0.029867,0.000000
50%,0.154181,0.154181,0.000000
75%,0.559046,0.559046,0.000000
max,50708.000000,1.092956,1.000000


In [10]:
# ============================================
# Ajuste del indicador de valores extremos
# ============================================

df["Utilization_extreme"] = (
    df["RevolvingUtilizationOfUnsecuredLines"] > utilization_p99
).astype(int)

print(
    f"Registros tratados como extremos: "
    f"{df['Utilization_extreme'].sum():,}"
)

print(
    f"Máximo de Utilization_clean: "
    f"{df['Utilization_clean'].max():.4f}"
)

Registros tratados como extremos: 1,500
Máximo de Utilization_clean: 1.0930


In [11]:
# ============================================
# Feature: historial de mora
# ============================================

df["total_late_count"] = (
    df["NumberOfTime30-59DaysPastDueNotWorse_imputed"]
    + df["NumberOfTimes90DaysLate_imputed"]
    + df["NumberOfTime60-89DaysPastDueNotWorse_imputed"]
)

# Indicador de cualquier atraso
df["any_late"] = (
    df["total_late_count"] > 0
).astype(int)

# Verificación
display(
    df[
        [
            "NumberOfTime30-59DaysPastDueNotWorse_imputed",
            "NumberOfTimes90DaysLate_imputed",
            "NumberOfTime60-89DaysPastDueNotWorse_imputed",
            "total_late_count",
            "any_late",
            "SeriousDlqin2yrs"
        ]
    ].head(10)
)

,NumberOfTime30-59DaysPastDueNotWorse_imputed,NumberOfTimes90DaysLate_imputed,NumberOfTime60-89DaysPastDueNotWorse_imputed,total_late_count,any_late,SeriousDlqin2yrs
0,2.0,0.0,0.0,2.0,1,1
1,0.0,0.0,0.0,0.0,0,0
2,1.0,1.0,0.0,2.0,1,0
3,0.0,0.0,0.0,0.0,0,0
4,1.0,0.0,0.0,1.0,1,0
5,0.0,0.0,0.0,0.0,0,0
6,0.0,0.0,0.0,0.0,0,0
7,0.0,0.0,0.0,0.0,0,0
8,0.0,0.0,0.0,0.0,0,0
9,0.0,0.0,0.0,0.0,0,0


In [12]:
# ============================================
# Features: severidad de mora
# ============================================

# Máximo número de atrasos observado
df["max_late_count"] = df[
    [
        "NumberOfTime30-59DaysPastDueNotWorse_imputed",
        "NumberOfTimes90DaysLate_imputed",
        "NumberOfTime60-89DaysPastDueNotWorse_imputed"
    ]
].max(axis=1)

# Indicador de mora severa (90+ días)
df["late_90plus"] = (
    df["NumberOfTimes90DaysLate_imputed"] > 0
).astype(int)

# Verificación
display(
    df[
        [
            "total_late_count",
            "max_late_count",
            "late_90plus",
            "SeriousDlqin2yrs"
        ]
    ].head(10)
)

,total_late_count,max_late_count,late_90plus,SeriousDlqin2yrs
0,2.0,2.0,0,1
1,0.0,0.0,0,0
2,2.0,1.0,1,0
3,0.0,0.0,0,0
4,1.0,1.0,0,0
5,0.0,0.0,0,0
6,0.0,0.0,0,0
7,0.0,0.0,0,0
8,0.0,0.0,0,0
9,0.0,0.0,0,0


In [13]:
# ============================================
# Features: utilización de crédito
# ============================================

# Indicador de utilización elevada
df["high_utilization"] = (
    df["Utilization_clean"] > 1
).astype(int)

# Indicador de utilización extrema
df["very_high_utilization"] = (
    df["Utilization_clean"] > 2
).astype(int)

# Categoría de utilización
df["utilization_band"] = pd.cut(
    df["Utilization_clean"],
    bins=[
        -np.inf,
        0.1,
        0.3,
        0.5,
        0.8,
        1.0,
        2.0,
        np.inf
    ],
    labels=[
        "very_low",
        "low",
        "moderate",
        "elevated",
        "high",
        "very_high",
        "extreme"
    ],
    include_lowest=True
)

# Verificación
display(
    df[
        [
            "Utilization_clean",
            "high_utilization",
            "very_high_utilization",
            "utilization_band",
            "SeriousDlqin2yrs"
        ]
    ].head(10)
)

,Utilization_clean,high_utilization,very_high_utilization,utilization_band,SeriousDlqin2yrs
0,0.766127,0,0,elevated,1
1,0.957151,0,0,high,0
2,0.658180,0,0,elevated,0
3,0.233810,0,0,low,0
4,0.907239,0,0,high,0
5,0.213179,0,0,low,0
6,0.305682,0,0,moderate,0
7,0.754464,0,0,elevated,0
8,0.116951,0,0,low,0
9,0.189169,0,0,low,0


In [14]:
# ============================================
# Feature: bandas de edad
# ============================================

df["age_band"] = pd.cut(
    df["age_clean"],
    bins=[17, 25, 35, 45, 55, 65, 75, np.inf],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "66-75",
        "76+"
    ],
    include_lowest=True
)

# Verificación
age_band_summary = (
    df.groupby("age_band", observed=False)["SeriousDlqin2yrs"]
    .agg(
        count="count",
        default_rate="mean"
    )
    .reset_index()
)

age_band_summary["default_rate_pct"] = (
    age_band_summary["default_rate"] * 100
)

display(age_band_summary)

,age_band,count,default_rate,default_rate_pct
0,18-25,3027,0.111662,11.166171
1,26-35,18458,0.111225,11.122548
2,36-45,29819,0.088132,8.813173
3,46-55,36704,0.075932,7.593178
4,56-65,33406,0.045830,4.583009
5,66-75,18470,0.026151,2.615051
6,76+,10116,0.020364,2.036378


In [15]:
# ============================================
# Feature: bandas de DebtRatio
# ============================================

df["debt_ratio_band"] = pd.cut(
    df["DebtRatio_clean"],
    bins=[
        -np.inf,
        0.1,
        0.3,
        0.5,
        1.0,
        2.0,
        10.0,
        np.inf
    ],
    labels=[
        "very_low",
        "low",
        "moderate",
        "elevated",
        "high",
        "very_high",
        "extreme"
    ],
    include_lowest=True
)

# Verificación
debt_band_summary = (
    df.groupby("debt_ratio_band", observed=False)["SeriousDlqin2yrs"]
    .agg(
        count="count",
        default_rate="mean"
    )
    .reset_index()
)

debt_band_summary["default_rate_pct"] = (
    debt_band_summary["default_rate"] * 100
)

display(debt_band_summary)

,debt_ratio_band,count,default_rate,default_rate_pct
0,very_low,24520,0.059299,5.929853
1,low,38041,0.058358,5.835809
2,moderate,31147,0.063602,6.360163
3,elevated,21155,0.098322,9.832191
4,high,4092,0.131720,13.172043
5,very_high,2168,0.065959,6.595941
6,extreme,28877,0.055719,5.571908


In [16]:
# ============================================
# Feature: indicador de MonthlyIncome faltante
# ============================================

df["income_missing"] = (
    df["MonthlyIncome"].isna()
).astype(int)

# Verificación
income_missing_summary = (
    df.groupby("income_missing")["SeriousDlqin2yrs"]
    .agg(
        count="count",
        default_rate="mean"
    )
    .reset_index()
)

income_missing_summary["default_rate_pct"] = (
    income_missing_summary["default_rate"] * 100
)

display(income_missing_summary)

,income_missing,count,default_rate,default_rate_pct
0,0,120269,0.069486,6.948590
1,1,29731,0.056137,5.613669


In [17]:
# ============================================
# Feature: severidad máxima de mora
# ============================================

df["late_severity"] = (
    df[
        [
            "NumberOfTime30-59DaysPastDueNotWorse_imputed",
            "NumberOfTimes90DaysLate_imputed",
            "NumberOfTime60-89DaysPastDueNotWorse_imputed"
        ]
    ]
    .max(axis=1)
)

# Clasificación de severidad
df["late_severity_band"] = pd.cut(
    df["late_severity"],
    bins=[-np.inf, 0, 1, 2, np.inf],
    labels=[
        "none",
        "low",
        "moderate",
        "high"
    ],
    include_lowest=True
)

# Resumen
late_severity_summary = (
    df.groupby(
        "late_severity_band",
        observed=False
    )["SeriousDlqin2yrs"]
    .agg(
        count="count",
        default_rate="mean"
    )
    .reset_index()
)

late_severity_summary["default_rate_pct"] = (
    late_severity_summary["default_rate"] * 100
)

display(late_severity_summary)

,late_severity_band,count,default_rate,default_rate_pct
0,none,119906,0.028447,2.844728
1,low,19791,0.145975,14.597544
2,moderate,5886,0.294597,29.459735
3,high,4417,0.450985,45.098483


In [18]:
# ============================================
# Feature: carga histórica de mora
# ============================================

df["late_burden"] = (
    df["total_late_count"]
    + 2 * df["late_90plus"]
)

# Clasificación de carga de mora
df["late_burden_band"] = pd.cut(
    df["late_burden"],
    bins=[-np.inf, 0, 1, 3, np.inf],
    labels=[
        "none",
        "low",
        "moderate",
        "high"
    ],
    include_lowest=True
)

# Resumen
late_burden_summary = (
    df.groupby(
        "late_burden_band",
        observed=False
    )["SeriousDlqin2yrs"]
    .agg(
        count="count",
        default_rate="mean"
    )
    .reset_index()
)

late_burden_summary["default_rate_pct"] = (
    late_burden_summary["default_rate"] * 100
)

display(late_burden_summary)

,late_burden_band,count,default_rate,default_rate_pct
0,none,119906,0.028447,2.844728
1,low,14911,0.104352,10.435249
2,moderate,8123,0.220116,22.011572
3,high,7060,0.463314,46.331445


In [19]:
# ============================================
# Feature: ingreso por línea de crédito
# ============================================

df["income_per_credit_line"] = (
    df["MonthlyIncome"] /
    df["NumberOfOpenCreditLinesAndLoans"].replace(0, np.nan)
)

# Restaurar 0 cuando no existen líneas de crédito abiertas
df["income_per_credit_line"] = (
    df["income_per_credit_line"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

income_credit_summary = pd.DataFrame({
    "metric": [
        "Mínimo",
        "P25",
        "Mediana",
        "P75",
        "P95",
        "P99",
        "Máximo"
    ],
    "value": [
        df["income_per_credit_line"].min(),
        df["income_per_credit_line"].quantile(0.25),
        df["income_per_credit_line"].median(),
        df["income_per_credit_line"].quantile(0.75),
        df["income_per_credit_line"].quantile(0.95),
        df["income_per_credit_line"].quantile(0.99),
        df["income_per_credit_line"].max()
    ]
})

display(income_credit_summary)

,metric,value
0,Mínimo,0.000000
1,P25,181.818182
2,Mediana,549.928571
3,P75,1000.000000
4,P95,2300.000000
5,P99,4600.000000
6,Máximo,300875.000000


In [20]:
# ============================================
# Tratamiento de valores extremos
# ============================================

income_credit_p99 = df["income_per_credit_line"].quantile(0.99)

df["income_per_credit_line_clean"] = (
    df["income_per_credit_line"]
    .clip(upper=income_credit_p99)
)

print(
    f"P99 de income_per_credit_line: "
    f"{income_credit_p99:,.2f}"
)

print(
    f"Máximo antes: "
    f"{df['income_per_credit_line'].max():,.2f}"
)

print(
    f"Máximo después: "
    f"{df['income_per_credit_line_clean'].max():,.2f}"
)

display(
    df[
        [
            "income_per_credit_line",
            "income_per_credit_line_clean"
        ]
    ].describe()
)

P99 de income_per_credit_line: 4,600.00
Máximo antes: 300,875.00
Máximo después: 4,600.00


,income_per_credit_line,income_per_credit_line_clean
count,150000.000000,150000.000000
mean,786.072351,739.664250
std,1846.912271,817.493178
min,0.000000,0.000000
25%,181.818182,181.818182
50%,549.928571,549.928571
75%,1000.000000,1000.000000
max,300875.000000,4600.000000


In [21]:
# ============================================
# Default rate por nivel de ingreso por línea
# ============================================

df["income_credit_band"] = pd.cut(
    df["income_per_credit_line_clean"],
    bins=[-np.inf, 200, 500, 1000, 2000, 5000, np.inf],
    labels=[
        "<=200",
        "200-500",
        "500-1000",
        "1000-2000",
        "2000-5000",
        ">5000"
    ],
    include_lowest=True
)

income_credit_default = (
    df.groupby(
        "income_credit_band",
        observed=False
    )["SeriousDlqin2yrs"]
    .agg(
        count="count",
        default_rate="mean"
    )
    .reset_index()
)

income_credit_default["default_rate_pct"] = (
    income_credit_default["default_rate"] * 100
)

display(income_credit_default)

,income_credit_band,count,default_rate,default_rate_pct
0,<=200,38791,0.064912,6.491196
1,200-500,31554,0.078152,7.815174
2,500-1000,43116,0.065080,6.508025
3,1000-2000,26575,0.057761,5.776105
4,2000-5000,9964,0.070353,7.035327
5,>5000,0,NaN,NaN


## Inventario de variables generadas

Durante la etapa de Feature Engineering se generaron las siguientes variables:

### Variables imputadas
- `MonthlyIncome_imputed`
- `NumberOfDependents_imputed`
- `NumberOfTime30-59DaysPastDueNotWorse_imputed`
- `NumberOfTimes90DaysLate_imputed`
- `NumberOfTime60-89DaysPastDueNotWorse_imputed`

### Variables derivadas de comportamiento de mora
- `total_late_count`
- `max_late_count`
- `late_90plus`
- `any_late`

### Variables de calidad y tratamiento
- `age_clean`
- `age_invalid`
- `DebtRatio_clean`
- `DebtRatio_extreme`
- `Utilization_clean`
- `Utilization_extreme`

### Variables agrupadas
- `age_band`
- `debt_ratio_band`
- `utilization_band`
- `late_severity_band`
- `late_burden_band`

### Variables de severidad y carga de mora
- `late_severity`
- `late_burden`

### Variables relacionadas con capacidad de crédito
- `income_per_credit_line`
- `income_per_credit_line_clean`

Estas variables serán evaluadas posteriormente en términos de capacidad predictiva,
redundancia, estabilidad y utilidad para el modelo de PD.

In [23]:
# ============================================
# Guardar dataset procesado
# ============================================

from pathlib import Path

OUTPUT_PATH = Path("../data/processed/pd_features.csv")

# Crear la carpeta si no existe
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Guardar dataset
df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Dataset procesado guardado correctamente.")
print(f"Ruta: {OUTPUT_PATH}")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")

Dataset procesado guardado correctamente.
Ruta: ..\data\processed\pd_features.csv
Filas: 150,000
Columnas: 48
